In [5]:
import pandas as pd

In [6]:
colunas_desejadas = [
    "Número Estação", "Entidade", "Empresa Estação", "Tecnologia", "Geração",
    "Latitude decimal", "Longitude decimal", "Município-UF", "UF"
]

df = pd.read_csv(
    "Estacoes_SMP.csv",
    sep=";",
    encoding="utf-8-sig",
    usecols=colunas_desejadas
)


/tmp/ipykernel_50117/3764823710.py:6: DtypeWarning: Columns (38: Município-UF) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


In [7]:
print(f"Number of stations: {len(df):,}")
df.head()

Number of stations: 3,331,248


,Número Estação,Entidade,Tecnologia,Latitude decimal,Longitude decimal,Empresa Estação,Geração,Município-UF,UF
0,683532154,TELEFONICA BRASIL S.A.,GSM,-20.142500,-44.898060,VIVO,2G,Divinópolis - MG,MG
1,689725370,TELEFONICA BRASIL S.A.,LTE,-20.147500,-44.891110,VIVO,4G,Divinópolis - MG,MG
2,687543665,CLARO S.A.,GSM,-20.177800,-44.898700,CLARO,2G,Divinópolis - MG,MG
3,665598939,TELEFONICA BRASIL S.A.,LTE,-20.142220,-44.888890,VIVO,4G,Divinópolis - MG,MG
4,689412746,TIM S A,LTE,-20.147861,-44.863056,TIM,4G,Divinópolis - MG,MG


## 1. Agrupamento por Ponto Físico Único (Torre / Site)

Em um mesmo local geográfico (mesma Latitude e Longitude), existem diversas antenas cadastradas: diferentes gerações (2G, 3G, 4G, 5G), tecnologias (GSM, LTE, NR) e até diferentes operadoras que compartilham a mesma torre (*colocation*).

Para lançar os cabos de rede (Modelo de Waxman), agrupamos todas as linhas de uma mesma coordenada em **um local físico único (1 nó por torre)**, consolidando quais tecnologias estão presentes em cada ponto.

In [8]:
# 1. Agrupar por coordenada geográfica única e consolidar as tecnologias
df_locais = (
    df.dropna(subset=["Latitude decimal", "Longitude decimal"])
    .groupby(["Latitude decimal", "Longitude decimal"])
    .agg(
        id_local=("Número Estação", "first"),
        qtd_antenas=("Número Estação", "count"),
        operadoras=("Empresa Estação", lambda x: ", ".join(sorted(set(x.dropna())))),
        geracoes=("Geração", lambda x: ", ".join(sorted(set(x.dropna())))),
        tecnologias=("Tecnologia", lambda x: ", ".join(sorted(set(x.dropna())))),
        tem_2g=("Geração", lambda x: "2G" in set(x)),
        tem_3g=("Geração", lambda x: "3G" in set(x)),
        tem_4g=("Geração", lambda x: "4G" in set(x)),
        tem_5g=("Geração", lambda x: "5G" in set(x))
    )
    .reset_index()
)

print(f"Total de torres físicas únicas: {len(df_locais)}")
print(f"Distribuição de tecnologias nas torres:")
print(f"- Torres com 2G: {df_locais['tem_2g'].sum()}")
print(f"- Torres com 3G: {df_locais['tem_3g'].sum()}")
print(f"- Torres com 4G: {df_locais['tem_4g'].sum()}")
print(f"- Torres com 5G: {df_locais['tem_5g'].sum()}")

# Exemplo das primeiras torres agrupadas:
display(df_locais[["Latitude decimal", "Longitude decimal", "operadoras", "geracoes", "tecnologias", "qtd_antenas"]].head(10))


Total de torres físicas únicas: 109933
Distribuição de tecnologias nas torres:
- Torres com 2G: 52442
- Torres com 3G: 69338
- Torres com 4G: 102392
- Torres com 5G: 59410


,Latitude decimal,Longitude decimal,operadoras,geracoes,tecnologias,qtd_antenas
0,-33.737780,-53.372500,VIVO,"2G, 3G, 4G","GSM, LTE, WCDMA",30
1,-33.690556,-53.458889,TIM,"2G, 3G, 4G","GSM, LTE, WCDMA",27
2,-33.690140,-53.455470,VIVO,"2G, 3G, 4G","GSM, LTE, WCDMA",45
3,-33.689300,-53.459500,CLARO,"2G, 3G, 4G","GSM, LTE, WCDMA",24
4,-33.666000,-53.265472,TIM,"2G, 3G","GSM, WCDMA",6
5,-33.665830,-53.265280,VIVO,"3G, 4G","LTE, WCDMA",27
6,-33.664170,-53.263610,VIVO,"2G, 3G, 4G","GSM, LTE, WCDMA",72
7,-33.663610,-53.263890,CLARO,"2G, 3G, 4G","GSM, LTE, WCDMA",15
8,-33.541520,-53.370920,VIVO,4G,LTE,3
9,-33.541110,-53.369720,VIVO,"2G, 3G, 4G","GSM, LTE, WCDMA",51


## 2. Implementação do Modelo de Waxman (1988)

A probabilidade de conexão entre duas estações $u$ e $v$ é dada por:
$$P(u, v) = \alpha \cdot e^{\frac{-d(u, v)}{\beta \cdot L}}$$

* **$d(u, v)$**: Distância real entre as estações (calculada pela fórmula de Haversine em km);
* **$L$**: Distância máxima entre quaisquer dois nós da rede ($L = \max d(u, v)$);
* **$\alpha \in (0, 1]$**: Controla a densidade geral da rede (maior $\alpha$ gera mais enlaces no total);
* **$\beta \in (0, 1]$**: Controla a proporção entre enlaces curtos e longos (menor $\beta$ força conexões estritamente locais).

In [ ]:
import numpy as np
import networkx as nx

def calcular_matriz_haversine(lats, lons):
    """Calcula a matriz de distâncias par-a-par em km usando a fórmula de Haversine vetorizada."""
    R = 6371.0  # Raio médio da Terra em km
    lat_rad = np.radians(lats)
    lon_rad = np.radians(lons)
    dlat = lat_rad[:, None] - lat_rad[None, :]
    dlon = lon_rad[:, None] - lon_rad[None, :]
    a = np.sin(dlat / 2.0)**2 + np.cos(lat_rad[:, None]) * np.cos(lat_rad[None, :]) * np.sin(dlon / 2.0)**2
    return 2.0 * R * np.arcsin(np.sqrt(np.clip(a, 0.0, 1.0)))

def gerar_grafo_waxman(df_pontos, alpha=0.4, beta=0.15, seed=42):
    """
    Gera um grafo de telecomunicações usando o Modelo de Waxman sobre os pontos físicos reais.
    """
    np.random.seed(seed)
    n = len(df_pontos)
    lats = df_pontos["Latitude decimal"].values
    lons = df_pontos["Longitude decimal"].values
    ids = df_pontos.index.values  # ou df_pontos["id_local"].values
    
    # 1. Distâncias reais e diâmetro máximo L
    dist_mat = calcular_matriz_haversine(lats, lons)
    L = np.max(dist_mat)
    print(f"Diâmetro geográfico máximo (L): {L:.2f} km")
    
    # 2. Probabilidade de Waxman para cada par
    prob_mat = alpha * np.exp(-dist_mat / (beta * L))
    
    # 3. Sorteio aleatório no triângulo superior (sem auto-laços nem arestas duplicadas)
    sorteio = np.random.rand(n, n)
    triangulo_superior = np.triu(np.ones((n, n), dtype=bool), k=1)
    conectar = (sorteio < prob_mat) & triangulo_superior
    
    # 4. Construção do grafo NetworkX com todos os atributos da torre
    G = nx.Graph()
    for idx, row in df_pontos.iterrows():
        G.add_node(
            idx,
            pos=(row["Longitude decimal"], row["Latitude decimal"]),
            lat=row["Latitude decimal"],
            lon=row["Longitude decimal"],
            id_estacao=row.get("id_local", ""),
            operadoras=row.get("operadoras", ""),
            geracoes=row.get("geracoes", ""),
            tecnologias=row.get("tecnologias", ""),
            qtd_antenas=row.get("qtd_antenas", 1),
            tem_5g=row.get("tem_5g", False)
        )
    
    u_indices, v_indices = np.where(conectar)
    for u, v in zip(u_indices, v_indices):
        dist_km = float(dist_mat[u, v])
        G.add_edge(ids[u], ids[v], weight=dist_km, distance_km=dist_km)
        
    return G

# Parâmetros de calibração do modelo Waxman
alpha = 0.4   # Densidade de cabos
beta = 0.15   # Proporção de cabos longos

G = gerar_grafo_waxman(df_locais, alpha=alpha, beta=beta, seed=42)
print(f"Grafo gerado: {G.number_of_nodes()} torres e {G.number_of_edges()} enlaces.")


Diâmetro geográfico máximo (L): 28.67 km
Grafo gerado: 1590 nós e 131657 arestas (enlaces).


## 3. Análise de Redes Complexas e Identificação de Gargalos

Com a topologia gerada, podemos calcular as métricas estruturais da rede e identificar **gargalos críticos** usando a **Centralidade de Intermediação (*Betweenness Centrality*)**:
Nós com alta intermediação funcionam como pontes vitais entre regiões da cidade. Se uma dessas torres falhar, a conectividade da malha é seriamente degradada.

In [ ]:
# 1. Métricas Globais da Rede
num_nos = G.number_of_nodes()
num_arestas = G.number_of_edges()
densidade = nx.density(G)
num_componentes = nx.number_connected_components(G)

print("=== MÉTRICAS GLOBAIS DA REDE ===")
print(f"Torres (Nós): {num_nos}")
print(f"Enlaces (Arestas): {num_arestas}")
print(f"Densidade: {densidade:.4f}")
print(f"Componentes Conexos: {num_componentes}")

# Maior componente conexa
maior_comp = G.subgraph(max(nx.connected_components(G), key=len)).copy()
print(f"Maior Componente: {len(maior_comp)} nós ({len(maior_comp)/num_nos*100:.1f}% da rede)")
print(f"Comprimento Médio do Caminho: {nx.average_shortest_path_length(maior_comp, weight='weight'):.2f} km")

# 2. Identificação de Gargalos (Betweenness Centrality ponderada pela distância)
betweenness = nx.betweenness_centrality(G, weight="weight")
top_gargalos = sorted(betweenness.items(), key=lambda x: x[1], reverse=True)[:5]

print("
=== TOP 5 GARGALOS (MAIOR BETWEENNESS CENTRALITY) ===")
for no_id, bc_score in top_gargalos:
    dados_no = G.nodes[no_id]
    operadoras = dados_no.get("operadoras", "N/A")
    geracoes = dados_no.get("geracoes", "N/A")
    grau = G.degree[no_id]
    print(f"Torre #{no_id} [Operadoras: {operadoras} | Gerações: {geracoes}]: Betweenness = {bc_score:.4f} | Grau = {grau} enlaces")


=== MÉTRICAS GLOBAIS DA REDE ===
Nós (Estações): 1590
Arestas (Enlaces): 131657
Densidade: 0.1042
Componentes Conexos: 1
Maior Componente: 1590 nós (100.0% da rede)


## 4. Visualização Geográfica da Rede

Desenhamos a malha de cabos sobre as coordenadas reais (Longitude x Latitude), destacando os nós com base na sua criticidade (centralidade de intermediação).

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 10))
pos = nx.get_node_attributes(G, "pos")

# 1. Desenhar enlaces (arestas) simulando cabos/rádios
nx.draw_networkx_edges(
    G, pos,
    alpha=0.20,
    edge_color="#1f77b4",
    width=0.8
)

# 2. Desenhar nós com cor baseada na centralidade de intermediação (plasma)
node_colors = [betweenness[node] for node in G.nodes()]
nodes_plot = nx.draw_networkx_nodes(
    G, pos,
    node_size=45,
    node_color=node_colors,
    cmap=plt.cm.plasma
)

cbar = plt.colorbar(nodes_plot, shrink=0.7)
cbar.set_label("Centralidade de Intermediação (Betweenness)", fontsize=11)

plt.title(
    f"Topologia Sintética de Telecomunicações - Modelo de Waxman\n{municipio_alvo} (α={alpha}, β={beta})",
    fontsize=14,
    pad=15
)
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.grid(True, linestyle="--", alpha=0.3)
plt.gca().set_aspect("equal")

plt.show()
